# 🍜 02. BeautifulSoup 완전정복
## — HTML에서 원하는 데이터만 골라내기

> **이 노트북에서 배울 것**
> 1. HTML 구조 복습
> 2. BeautifulSoup 기본 4가지 함수
> 3. 텍스트 & 속성값 추출
> 4. None 안전 처리 패턴
> 5. 개발자도구로 선택자 찾는 법
> 6. 실전: books.toscrape.com 완전 공략


## 1. HTML 구조 복습

웹페이지는 HTML 태그들이 겹겹이 쌓인 구조예요.

```html
<html>
  <body>
    <div class="product-list">          ← 컨테이너 (상자)
      <div class="product-item">        ← 상품 카드 1
        <span class="name">립밤</span>  ← 이름
        <span class="price">12,000원</span>  ← 가격
      </div>
      <div class="product-item">        ← 상품 카드 2
        <span class="name">선크림</span>
        <span class="price">25,000원</span>
      </div>
    </div>
  </body>
</html>
```

**핵심 개념:**
- `태그`: `<div>`, `<span>`, `<p>`, `<a>` 등
- `class`: 태그에 붙은 이름표 (CSS 선택자에 `.` 사용)
- `id`: 유일한 이름표 (CSS 선택자에 `#` 사용)
- `href`, `src`: 태그의 속성값


In [1]:
# HTML 문자열로 BeautifulSoup 연습
from bs4 import BeautifulSoup

# 가짜 HTML (연습용)
html = """
<html>
<body>
  <div class="product-list">
    <div class="product-item">
      <span class="name">립밤 촉촉이</span>
      <span class="price">12,000원</span>
      <a href="/products/001" class="detail-link">상세보기</a>
    </div>
    <div class="product-item">
      <span class="name">수분 선크림</span>
      <span class="price">25,000원</span>
      <a href="/products/002" class="detail-link">상세보기</a>
    </div>
    <div class="product-item">
      <span class="name">촉촉 에센스</span>
      <span class="price">38,000원</span>
      <a href="/products/003" class="detail-link">상세보기</a>
    </div>
  </div>
</body>
</html>
"""

soup = BeautifulSoup(html, 'html.parser')
print("파싱 완료!")
print(type(soup))  # BeautifulSoup 객체


파싱 완료!
<class 'bs4.BeautifulSoup'>


## 2. 핵심 함수 ① — `find()` : 첫 번째 하나만 찾기

```python
soup.find('태그이름')
soup.find('태그이름', class_='클래스명')
soup.find('태그이름', id='아이디')
```

> ⚠️ `class`는 파이썬 예약어라서 `class_`로 씁니다!


In [2]:
# find() 실습 — 첫 번째 요소만 가져옴

# 태그이름으로 찾기
first_div = soup.find('div')
print("첫 번째 div:")
print(first_div)
print()

# 태그 + 클래스로 찾기
first_item = soup.find('div', class_='product-item')
print("첫 번째 product-item:")
print(first_item)


첫 번째 div:
<div class="product-list">
<div class="product-item">
<span class="name">립밤 촉촉이</span>
<span class="price">12,000원</span>
<a class="detail-link" href="/products/001">상세보기</a>
</div>
<div class="product-item">
<span class="name">수분 선크림</span>
<span class="price">25,000원</span>
<a class="detail-link" href="/products/002">상세보기</a>
</div>
<div class="product-item">
<span class="name">촉촉 에센스</span>
<span class="price">38,000원</span>
<a class="detail-link" href="/products/003">상세보기</a>
</div>
</div>

첫 번째 product-item:
<div class="product-item">
<span class="name">립밤 촉촉이</span>
<span class="price">12,000원</span>
<a class="detail-link" href="/products/001">상세보기</a>
</div>


## 3. 핵심 함수 ② — `find_all()` : 조건에 맞는 전부 찾기

```python
soup.find_all('태그이름')          # 해당 태그 모두
soup.find_all('태그이름', class_='클래스명')  # 조건 맞는 모두
```

결과: **리스트** (0개 이상의 요소들)


In [3]:
# find_all() 실습 — 전부 가져옴

# 모든 product-item 찾기
all_items = soup.find_all('div', class_='product-item')
print(f"총 {len(all_items)}개의 상품 발견")
print()

# 반복문으로 각 상품 처리
for i, item in enumerate(all_items, 1):
    name = item.find('span', class_='name')
    price = item.find('span', class_='price')
    print(f"{i}번 상품: {name.text} — {price.text}")


총 3개의 상품 발견

1번 상품: 립밤 촉촉이 — 12,000원
2번 상품: 수분 선크림 — 25,000원
3번 상품: 촉촉 에센스 — 38,000원


## 4. 핵심 함수 ③ — `select()` : CSS 선택자로 찾기 ⭐

가장 많이 쓰는 방법! 개발자도구의 선택자를 그대로 사용할 수 있어요.

```python
soup.select('div')              # 태그만
soup.select('.product-item')    # 클래스만 (앞에 .)
soup.select('#header')          # ID (앞에 #)
soup.select('div.product-item') # 태그 + 클래스
soup.select('div > span')       # div 바로 아래 span
soup.select('div span')         # div 안의 모든 span
soup.select('a[href]')          # href 속성이 있는 a
soup.select('a[href*="product"]') # href에 "product"가 들어간 a
```

결과: **리스트** (find_all과 같음)


In [4]:
# select() 실습

# 클래스로 찾기
items = soup.select('.product-item')
print(f"select('.product-item'): {len(items)}개 발견")

# 태그 + 클래스
names = soup.select('span.name')
print(f"\nselect('span.name'): {len(names)}개 발견")
for name in names:
    print(f"  - {name.text}")

# 속성 조건
links = soup.select('a[href*="products"]')
print(f"\nselect('a[href*=products]'): {len(links)}개 발견")
for link in links:
    print(f"  - {link['href']}")


select('.product-item'): 3개 발견

select('span.name'): 3개 발견
  - 립밤 촉촉이
  - 수분 선크림
  - 촉촉 에센스

select('a[href*=products]'): 3개 발견
  - /products/001
  - /products/002
  - /products/003


## 5. 핵심 함수 ④ — `select_one()` : CSS 선택자로 하나만

`select()`의 결과 중 첫 번째 하나만 가져와요.
`find()`와 같은 역할이지만 CSS 선택자를 쓸 수 있어요.

```python
soup.select_one('.product-item')  # 첫 번째 하나
item.select_one('.name')          # item 안에서 찾기
```


In [5]:
# select_one() 실습

# 전체 soup에서 하나
first_item = soup.select_one('.product-item')
print("첫 번째 상품:")
print(first_item)

# 하나를 찾은 뒤 그 안에서 다시 찾기
name = first_item.select_one('.name')
price = first_item.select_one('.price')
print(f"\n이름: {name.text}")
print(f"가격: {price.text}")


첫 번째 상품:
<div class="product-item">
<span class="name">립밤 촉촉이</span>
<span class="price">12,000원</span>
<a class="detail-link" href="/products/001">상세보기</a>
</div>

이름: 립밤 촉촉이
가격: 12,000원


## 6. 텍스트 추출 — `.text` vs `.get_text()`

| 방법 | 특징 |
|------|------|
| `.text` | 태그 안 모든 텍스트 (자식 태그 포함) |
| `.get_text()` | `.text`와 동일, 옵션 지정 가능 |
| `.get_text(strip=True)` | 앞뒤 공백 자동 제거 |
| `.text.strip()` | 앞뒤 공백 수동 제거 |

```python
# 두 방법 모두 자주 쓰임
element.text.strip()
element.get_text(strip=True)
```


In [6]:
# 텍스트 추출 실습

item = soup.select_one('.product-item')

# 태그 전체 텍스트 (공백 포함)
print("=== .text (원본) ===")
print(repr(item.text))

print("\n=== .text.strip() ===")
print(item.text.strip())

print("\n=== .get_text(strip=True) ===")
print(item.get_text(strip=True))

# 개별 요소 텍스트
name_el = item.select_one('.name')
print(f"\n이름 텍스트: '{name_el.text}'")
print(f"이름 strip: '{name_el.text.strip()}'")


=== .text (원본) ===
'\n립밤 촉촉이\n12,000원\n상세보기\n'

=== .text.strip() ===
립밤 촉촉이
12,000원
상세보기

=== .get_text(strip=True) ===
립밤 촉촉이12,000원상세보기

이름 텍스트: '립밤 촉촉이'
이름 strip: '립밤 촉촉이'


## 7. 속성값 추출 — `['속성명']` vs `.get('속성명')`

링크(`href`), 이미지(`src`), 데이터 속성(`data-id`) 등을 가져올 때 사용해요.

```python
element['href']          # 없으면 KeyError 발생
element.get('href')      # 없으면 None 반환 (안전)
element.get('href', '')  # 없으면 '' 반환 (기본값 지정)
```


In [7]:
# 속성값 추출 실습

# a 태그의 href 속성 가져오기
link = soup.select_one('a.detail-link')
print(f"태그 전체: {link}")

# 방법 1: 딕셔너리처럼 접근
href1 = link['href']
print(f"\n['href']: {href1}")

# 방법 2: .get() — 더 안전
href2 = link.get('href')
print(f".get('href'): {href2}")

# 모든 링크의 href 가져오기
all_links = soup.select('a[href]')
print("\n--- 모든 링크 ---")
for a in all_links:
    print(f"  텍스트: {a.text.strip()!r:15}  href: {a.get('href')}")


태그 전체: <a class="detail-link" href="/products/001">상세보기</a>

['href']: /products/001
.get('href'): /products/001

--- 모든 링크 ---
  텍스트: '상세보기'           href: /products/001
  텍스트: '상세보기'           href: /products/002
  텍스트: '상세보기'           href: /products/003


## 8. ⚠️ 가장 중요한 패턴 — None 안전 처리

크롤링에서 가장 많이 나는 에러:
```
AttributeError: 'NoneType' object has no attribute 'text'
```

원인: 요소를 찾지 못하면 `None`이 반환되는데, `None.text`를 호출할 수 없어요.

**해결법 3가지:**
```python
# 방법 1: if else (가장 많이 씀)
name = el.text.strip() if el else None

# 방법 2: try-except
try:
    name = el.text.strip()
except:
    name = None

# 방법 3: and 연산자
name = el and el.text.strip()
```


In [8]:
# None 안전 처리 실습

# 존재하지 않는 클래스 선택
missing_el = soup.select_one('.존재하지않는클래스')
print(f"못 찾으면: {missing_el}")  # None

# ❌ 잘못된 방법 — 에러 발생!
try:
    name = missing_el.text  # AttributeError!
except AttributeError as e:
    print(f"\n에러 발생: {e}")

# ✅ 올바른 방법 — 항상 이렇게!
name = missing_el.text.strip() if missing_el else None
print(f"\n안전한 방법 결과: {name}")

# 실제 사용 예
for item in soup.select('.product-item'):
    name = item.select_one('.name')
    price = item.select_one('.price')
    rating = item.select_one('.rating')  # 이 클래스는 없음!

    print({
        '이름': name.text.strip() if name else None,
        '가격': price.text.strip() if price else None,
        '평점': rating.text.strip() if rating else '정보없음',  # 기본값 지정
    })


못 찾으면: None

에러 발생: 'NoneType' object has no attribute 'text'

안전한 방법 결과: None
{'이름': '립밤 촉촉이', '가격': '12,000원', '평점': '정보없음'}
{'이름': '수분 선크림', '가격': '25,000원', '평점': '정보없음'}
{'이름': '촉촉 에센스', '가격': '38,000원', '평점': '정보없음'}


## 9. 개발자도구로 선택자 찾는 법

실제 사이트에서 어떤 선택자를 써야 하는지 찾는 방법이에요.

### 방법 1: Copy Selector (가장 빠름)
```
1. 크롬에서 원하는 요소 위에 마우스 올리기
2. 우클릭 → "검사" (Inspect)
3. 파란색으로 강조된 태그에서 우클릭
4. Copy → Copy selector
```

### 방법 2: 직접 읽기 (이해하기 좋음)
```html
<ul class="book-list">          → soup.select('ul.book-list')
  <li class="book-item">        → soup.select('li.book-item')
    <a href="/book/1">          → soup.select('li.book-item a')
      <h3>책 제목</h3>          → soup.select('li.book-item h3')
    </a>
  </li>
</ul>
```

### 방법 3: Elements 탭 검색 (Ctrl+F)
```
F12 → Elements 탭 → Ctrl+F → 클래스명 검색
```


## 10. 실전 실습 — books.toscrape.com 완전 공략

이제 배운 것을 실제 사이트에 적용해봅시다!

**books.toscrape.com HTML 구조:**
```html
<article class="product_pod">
  <p class="star-rating Three">   ← 평점 (class에 숫자 단어)
  <h3>
    <a href="..." title="책 제목">  ← 제목은 title 속성에!
  </h3>
  <p class="price_color">£51.77</p>  ← 가격
</article>
```


In [10]:
# 실전: books.toscrape.com 1페이지 수집
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = 'http://books.toscrape.com/'
headers = {'User-Agent': 'Mozilla/5.0'}

response = requests.get(url, headers=headers)
response.encoding = 'utf-8'
print(f"상태 코드: {response.status_code}")

soup = BeautifulSoup(response.text, 'html.parser')


상태 코드: 200


In [11]:
# 페이지 구조 탐색
# (개발자도구 없이도 파악할 수 있어요)

# 1. 책 컨테이너 확인
books = soup.select('article.product_pod')
print(f"총 {len(books)}권 발견")
print()

# 2. 첫 번째 책의 HTML 구조 확인
first_book = books[0]
print("=== 첫 번째 책 HTML ===")
print(first_book.prettify()[:600])


총 20권 발견

=== 첫 번째 책 HTML ===
<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_pric


In [12]:
# 데이터 추출

rating_map = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
rows = []

for book in books:
    # 제목: h3 > a 태그의 title 속성
    title_el = book.select_one('h3 a')
    title = title_el['title'] if title_el else None
    # ↑ title은 태그 안 텍스트가 아니라 속성에 있어요!

    # 가격: p.price_color의 텍스트
    price_el = book.select_one('p.price_color')
    price = price_el.text.strip() if price_el else None

    # 평점: p.star-rating의 두 번째 class가 숫자 단어
    # class 예: ['star-rating', 'Three']
    rating_el = book.select_one('p.star-rating')
    if rating_el:
        rating_word = rating_el['class'][1]  # 'Three'
        rating = rating_map.get(rating_word, 0)
    else:
        rating = None

    # 링크
    link_el = book.select_one('h3 a')
    link = 'http://books.toscrape.com/catalogue/' + link_el['href'] if link_el else None

    rows.append({
        '제목': title,
        '가격': price,
        '평점': rating,
        '링크': link,
    })

df = pd.DataFrame(rows)
print(df[['제목', '가격', '평점']].to_string(index=False))


                                                                                            제목     가격  평점
                                                                          A Light in the Attic £51.77   3
                                                                            Tipping the Velvet £53.74   1
                                                                                    Soumission £50.10   1
                                                                                 Sharp Objects £47.82   4
                                                         Sapiens: A Brief History of Humankind £54.23   5
                                                                               The Requiem Red £22.65   1
                                            The Dirty Little Secrets of Getting Your Dream Job £33.34   4
       The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull £17.93   3
The Boys in the Boat: Nine Americans and Their

In [13]:
# 여러 페이지 수집 (3페이지)
import time

all_books = []

for page in range(1, 4):
    url = f'http://books.toscrape.com/catalogue/page-{page}.html'
    response = requests.get(url, headers=headers)
    response.encoding = 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')
    books = soup.select('article.product_pod')

    for book in books:
        title_el = book.select_one('h3 a')
        price_el = book.select_one('p.price_color')
        rating_el = book.select_one('p.star-rating')

        all_books.append({
            '제목': title_el['title'] if title_el else None,
            '가격': price_el.text.strip() if price_el else None,
            '평점': rating_map.get(
                rating_el['class'][1] if rating_el else '', 0
            ),
            '페이지': page,
        })

    print(f"  {page}페이지 완료 ({len(books)}권)")
    time.sleep(1)  # ← 항상 1초 쉬기!

df_all = pd.DataFrame(all_books)
print(f"\n총 {len(df_all)}권 수집!")
df_all.head(5)


  1페이지 완료 (20권)
  2페이지 완료 (20권)
  3페이지 완료 (20권)

총 60권 수집!


,제목,가격,평점,페이지
0,A Light in the Attic,£51.77,3,1
1,Tipping the Velvet,£53.74,1,1
2,Soumission,£50.10,1,1
3,Sharp Objects,£47.82,4,1
4,Sapiens: A Brief History of Humankind,£54.23,5,1


In [ ]:
# 연습문제 1: 5점짜리 책만 필터링

five_star = df_all[df_all['평점'] == 5]
print(f"5점 책 총 {len(five_star)}권:")
print(five_star[['제목', '가격']].to_string(index=False))


In [ ]:
# 연습문제 2: 평점별 평균 가격 (문자열 → 숫자 변환 필요)

df_all['가격_숫자'] = (df_all['가격']
    .str.replace('£', '')
    .str.strip()
    .pipe(pd.to_numeric, errors='coerce')
)

print("=== 평점별 평균 가격 ===")
result = df_all.groupby('평점')['가격_숫자'].agg(['mean', 'count'])
result.columns = ['평균가격(£)', '책수']
result['평균가격(£)'] = result['평균가격(£)'].round(2)
print(result.to_string())


## ✅ 오늘 배운 것 정리

| 함수 | 언제 쓰나 | 반환 |
|------|----------|------|
| `find('태그')` | 첫 번째 하나 | 요소 or None |
| `find_all('태그')` | 전부 | 리스트 |
| `select('선택자')` | CSS로 전부 | 리스트 |
| `select_one('선택자')` | CSS로 하나 | 요소 or None |
| `el.text.strip()` | 텍스트 추출 | 문자열 |
| `el['속성']` | 속성값 | 문자열 |
| `el.get('속성')` | 속성값 (안전) | 문자열 or None |

**핵심 패턴:**
```python
element = soup.select_one('선택자')
value = element.text.strip() if element else None  # 항상!
```

> **다음 노트북**: 사람인 채용공고 실전 크롤링!
